# Generating Shakespearean Text Using a Character RNN


In [1]:
import tensorflow as tf
from tensorflow import keras


In [2]:
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))
tf.config.get_visible_devices()

Num GPUs Available:  0


[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]

In [3]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [4]:
# Ensure TensorFlow uses the GPU
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(f'RuntimeError: {e}')

In [5]:
shakespeare_url = "https://homl.info/shakespeare" # shortcut URL
filepath = keras.utils.get_file("shakespeare.txt", shakespeare_url)
with open(filepath) as f:
  shakespeare_text = f.read()


1115394/1115394 [==============================] - 0s 0us/step


In [6]:
shakespeare_text[0:100]
#vemos los primeros 100 caracteres

'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'

In [7]:
tokenizer = keras.preprocessing.text.Tokenizer(char_level=True)
tokenizer.fit_on_texts([shakespeare_text])
#convertimos el texto en secuencias de números

In [8]:
tokenizer.texts_to_sequences(["The House"])
#convierte una palabra en una secuencia de números

[[3, 7, 2, 1, 7, 4, 14, 8, 2]]

In [9]:
tokenizer.sequences_to_texts([[3, 7, 2, 1, 7, 4, 14, 8, 2]])
#Así vuelve a ser texto

['t h e   h o u s e']

In [10]:
max_id = len(tokenizer.word_index)
max_id
#cuantos caracteres distintos hay

39

In [11]:
import numpy as np

[encoded] = np.array(tokenizer.texts_to_sequences([shakespeare_text])) - 1

In [12]:
encoded[0:100]

array([19,  5,  8,  7,  2,  0, 18,  5,  2,  5, 35,  1,  9, 23, 10, 21,  1,
       19,  3,  8,  1,  0, 16,  1,  0, 22,  8,  3, 18,  1,  1, 12,  0,  4,
        9, 15,  0, 19, 13,  8,  2,  6,  1,  8, 17,  0,  6,  1,  4,  8,  0,
       14,  1,  0,  7, 22,  1,  4, 24, 26, 10, 10,  4, 11, 11, 23, 10,  7,
       22,  1,  4, 24, 17,  0,  7, 22,  1,  4, 24, 26, 10, 10, 19,  5,  8,
        7,  2,  0, 18,  5,  2,  5, 35,  1,  9, 23, 10, 15,  3, 13])

In [13]:
dataset_size = len(encoded)

In [14]:
#Split into training, validation and test
train_size = dataset_size * 90 // 100
dataset = tf.data.Dataset.from_tensor_slices(encoded[:train_size])
print("training is")
print(train_size)

training is
1003854


El problema es que no tenemos muchas entradas, muchos registros como tenemos habitualmente sino una larga lista de caracteres en un solo array.
Pero tenemos un método, llamado windows() que usaremos para convertir esta larga secuencia de caracteres (o números) en muchas ***ventanas*** de texto más pequeñas.
Cada instancia de datos será una subcadena bastante corta del texto completo, y el RNR utilizará solo esta cadena en cada paso de entrenamiento. Esto se llama retropropagación (*backpropagation*) truncada a través del tiempo.

Llamemos al método **window()** para crear un conjunto de datos de ventanas de texto corto:

In [15]:
n_steps = 100
window_length = n_steps + 1 # target = input shifted 1 character ahead
dataset = dataset.window(window_length, shift=1, drop_remainder=True)


In [16]:
dataset

<_WindowDataset element_spec=DatasetSpec(TensorSpec(shape=(), dtype=tf.int64, name=None), TensorShape([]))>

In [17]:
dataset = dataset.flat_map(lambda window: window.batch(window_length))
#Esta función flat_map lo que hará es convertir datasets anidados,
# como este  {{1, 2}, {3, 4, 5, 6}} en un dataset plano d de tensores de tamaño 2
#  {[1, 2], [3, 4], [5, 6]}
#En este caso, cada datset tendrá ahora, ventanas consecutivas de 101 caracteres

In [18]:
dataset

<_FlatMapDataset element_spec=TensorSpec(shape=(None,), dtype=tf.int64, name=None)>

In [19]:
#seteamos el tamaño del batch size para el entrenamiento
batch_size = 32

#mezclamos las ventanas, para que no haya una correlación directa entre ellas
dataset = dataset.shuffle(10000).batch(batch_size) #mezclamos las ventanas
dataset = dataset.map(lambda windows: (windows[:, :-1], windows[:, 1:])) #transforma cada ventana en un par de entrada-salida


In [20]:
dataset

<_MapDataset element_spec=(TensorSpec(shape=(None, None), dtype=tf.int64, name=None), TensorSpec(shape=(None, None), dtype=tf.int64, name=None))>

In [21]:
dataset = dataset.map(
lambda X_batch, Y_batch: (tf.one_hot(X_batch, depth=max_id), Y_batch))


In [22]:
dataset

<_MapDataset element_spec=(TensorSpec(shape=(None, None, 39), dtype=tf.float32, name=None), TensorSpec(shape=(None, None), dtype=tf.int64, name=None))>

In [ ]:
dataset = dataset.prefetch(1)


In [ ]:
model = keras.models.Sequential([
    keras.layers.GRU(128, return_sequences=True, input_shape=[None, max_id],
                     dropout=0.2,
                     recurrent_dropout=0.2
                     ),
    keras.layers.GRU(128, return_sequences=True,
                     dropout=0.2,
                     recurrent_dropout=0.2
                     ),
    keras.layers.TimeDistributed(keras.layers.Dense(max_id,
                                                    activation="softmax"))
])
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam")
history = model.fit(dataset, epochs=20)


Epoch 1/20
31368/31368 [==============================] - 457s 14ms/step - loss: 1.6198
Epoch 2/20
31368/31368 [==============================] - 437s 14ms/step - loss: 1.5364
Epoch 3/20
31368/31368 [==============================] - 463s 15ms/step - loss: 1.5136
Epoch 4/20
31368/31368 [==============================] - 454s 14ms/step - loss: 1.5021
Epoch 5/20
31368/31368 [==============================] - 442s 14ms/step - loss: 1.4946
Epoch 6/20
24465/31368 [======================>.......] - ETA: 1:40 - loss: 1.4874

KeyboardInterrupt: 

In [ ]:
model.save('rnr_shakespeare_text_generator.h5')

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [ ]:
# Recrea exactamente el mismo modelo solo desde el archivo
model = keras.models.load_model('rnr_shakespeare_text_generator.h5')

In [ ]:
def preprocess(texts):
  X = np.array(tokenizer.texts_to_sequences(texts)) - 1
  return tf.one_hot(X, max_id)

def posprocess(pred):
  return tokenizer.sequences_to_texts( [[ np.argmax(pred[0][-1])+1]])

X_new = preprocess(["How are yo"])
Y_pred = model.predict(X_new)
posprocess(Y_pred)



1/1 [==============================] - 1s 641ms/step


['u']

**Versión simple, siempre tomando al caracter más probable**

In [ ]:
def next_char_simple(text):
  X_new = preprocess(text)
  Y_pred = model.predict(X_new)
  return posprocess(Y_pred)[0]

def complete_text_simple(text, n_chars=50):
  for _ in range(n_chars):
    text += next_char_simple(text)
  return text

complete_text_simple("t")

2/2 [==============================] - 0s 5ms/step


'thhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhh'

In [ ]:
complete_text_simple("the shadow is")

2/2 [==============================] - 0s 6ms/step


'the shadow ishhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhhh'

**Versión mejorada, tomando caracteres con diversa probabilidad. Generará textos más originales**

In [ ]:
def next_char(text, temperature=1):
  X_new = preprocess([text])
  y_proba = model.predict(X_new)[0, -1:, :]
  rescaled_logits = tf.math.log(y_proba) / temperature
  char_id = tf.random.categorical(rescaled_logits, num_samples=1) + 1
  return tokenizer.sequences_to_texts(char_id.numpy())[0]

def complete_text(text, n_chars=50, temperature=1):
  for _ in range(n_chars):
    text += next_char(text, temperature)
  return text

print(complete_text("t", temperature=0.2))



1/1 [==============================] - 0s 31ms/step
t the storm of the pretty ones of the sealow of the


In [ ]:
print(complete_text("the shadow is ", temperature=0.2))


1/1 [==============================] - 0s 19ms/step
the shadow is the sea of them the sheep-shands of the sheep-shea
